# Module Loading & Environment Setup

In [5]:
import os
import time
import pickle
import shutil
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy.ndimage import median_filter

# ==========================================
# 1. KONFIGURASI DIREKTORI
# ==========================================
DATA_DIR = '/kaggle/input/competitions/ariel-data-challenge-2025'
SAVE_BASE_DIR = '/kaggle/working/processed_data'

if not os.path.exists(DATA_DIR):
    raise FileNotFoundError(f"Dataset tidak ditemukan di {DATA_DIR}.")

# ==========================================
# 2. FUNGSI PRAPEMROSESAN UTAMA
# ==========================================
def adc_conversion(data, instrument, adc_info_df):
    gain = adc_info_df[f'{instrument}_adc_gain'].values[0]
    offset = adc_info_df[f'{instrument}_adc_offset'].values[0]
    return (data.astype(np.float64) / gain) + offset

def remove_cosmic_rays(data_cube, threshold=5.0, size=3):
    median_filtered = median_filter(data_cube, size=(size, 1, 1))
    diff = np.abs(data_cube - median_filtered)
    mad = np.median(diff) + 1e-6
    outliers = diff > (threshold * mad)
    return np.where(outliers, median_filtered, data_cube)

def process_single_observation(planet_id, obs_idx, adc_info_df, fgs_bin_size, is_train=True):
    # Mengonversi planet_id ke string tanpa desimal
    p_id_str = str(int(planet_id))
    split = 'train' if is_train else 'test'
    base_path = os.path.join(DATA_DIR, split, p_id_str)
    
    result = {'planet_id': planet_id, 'obs_idx': obs_idx, 'fgs_signal': None, 'airs_signal': None}
    
    # Proses FGS1
    fgs_path = os.path.join(base_path, f'FGS1_signal_{obs_idx}.parquet')
    if os.path.exists(fgs_path):
        fgs_raw = pq.read_table(fgs_path).to_pandas().values.reshape(135000, 32, 32)
        fgs_photons = adc_conversion(fgs_raw, 'FGS1', adc_info_df)
        fgs_clean = remove_cosmic_rays(fgs_photons)
        # Reshape dinamis berdasarkan fgs_bin_size
        fgs_time_binned = fgs_clean.reshape(135000 // fgs_bin_size, fgs_bin_size, 32, 32).sum(axis=1)
        result['fgs_signal'] = fgs_time_binned.sum(axis=(1, 2))
        
    # Proses AIRS-CH0
    airs_path = os.path.join(base_path, f'AIRS-CH0_signal_{obs_idx}.parquet')
    if os.path.exists(airs_path):
        airs_raw = pq.read_table(airs_path).to_pandas().values.reshape(11250, 32, 356)
        airs_photons = adc_conversion(airs_raw, 'AIRS-CH0', adc_info_df)
        airs_clean = remove_cosmic_rays(airs_photons)
        airs_time_binned = airs_clean.reshape(11250 // 5, 5, 32, 356).sum(axis=1)
        result['airs_signal'] = airs_time_binned.sum(axis=1)
        
    return result

# ==========================================
# 3. PEMROSESAN MASSAL (REVISI STRUKTUR)
# ==========================================
def process_and_save_all(meta_df, adc_info_df, fgs_bin_size, is_train=True):
    split_name = 'train' if is_train else 'test'
    # Menggunakan struktur folder yang sinkron dengan jadwal master
    save_dir = os.path.join('/kaggle/working/', f'processed_data_bin_{fgs_bin_size}')
    
    if os.path.exists(save_dir):
        shutil.rmtree(save_dir)
    os.makedirs(save_dir, exist_ok=True)
    
    print(f"Memproses dataset {split_name} (Binning FGS: {fgs_bin_size})")
    
    # Hapus [:5] jika ingin memproses seluruh dataset
    for index, row in meta_df[:5].iterrows():
        planet_id = int(row['planet_id'])
        start_time = time.time()
        
        try:
            processed_data = process_single_observation(
                planet_id=planet_id, obs_idx=0, adc_info_df=adc_info_df,
                fgs_bin_size=fgs_bin_size, is_train=is_train
            )
            
            save_path = os.path.join(save_dir, f"{split_name}_{planet_id}.pkl")
            with open(save_path, 'wb') as f:
                pickle.dump(processed_data, f)
            print(f"ID {planet_id} selesai ({time.time() - start_time:.1f}s)")
        except Exception as e:
            print(f"ID {planet_id} gagal: {str(e)}")

# ==========================================
# 4. EKSEKUSI
# ==========================================
adc_info = pd.read_csv(os.path.join(DATA_DIR, 'adc_info.csv'))
train_meta = pd.read_csv(os.path.join(DATA_DIR, 'train_star_info.csv'))

binning_list = [25, 50, 75]
for bin_size in binning_list:
    process_and_save_all(train_meta, adc_info, fgs_bin_size=bin_size, is_train=True)

print("Tahap 1 selesai. Semua data tersimpan di folder binning masing-masing.")

Memproses dataset train (Binning FGS: 25)
ID 34983 selesai (23.5s)
ID 1873185 selesai (23.7s)
ID 3849793 selesai (23.9s)
ID 8456603 selesai (23.4s)
ID 23615382 selesai (25.2s)
Memproses dataset train (Binning FGS: 50)
ID 34983 selesai (23.0s)
ID 1873185 selesai (23.5s)
ID 3849793 selesai (23.5s)
ID 8456603 selesai (23.2s)
ID 23615382 selesai (24.6s)
Memproses dataset train (Binning FGS: 75)
ID 34983 selesai (24.1s)
ID 1873185 selesai (24.7s)
ID 3849793 selesai (24.5s)
ID 8456603 selesai (23.7s)
ID 23615382 selesai (24.1s)
Tahap 1 selesai. Semua data tersimpan di folder binning masing-masing.


# Data Loading & Fast Mode Configuration

In [6]:
# ==========================================
# TAHAP 2: DATA LOADING & FAST MODE CONFIGURATION
# ==========================================

print("Memuat data metadata dan konfigurasi...")

# 1. Memuat seluruh dataset pelatihan dan pengujian menggunakan loader dari library Juara 1
# Fungsi load_all_train_data() membaca train_star_info.csv dan data dasar lainnya
train_data = kgs.load_all_train_data()
test_data = kgs.load_all_test_data()

# 2. Konfigurasi Fast Mode
# Jika Anda sedang bereksperimen, set fast_mode = True agar hanya memproses 20 planet.
# Ini menghemat waktu eksekusi hingga 90% selama fase debugging.
fast_mode = True  # Ubah ke False jika sudah siap melakukan submission full dataset

if fast_mode and not kgs.is_submission:
    print("Fast Mode aktif: Hanya memproses 20 planet untuk pengujian.")
    train_data = train_data[:20]
    # Kita menggunakan copy agar tidak merusak data aslinya
    test_data = copy.deepcopy(train_data)
else:
    print("Mode Penuh: Memproses seluruh dataset kompetisi.")

print(f"Dataset dimuat. Total planet untuk diproses: {len(train_data)}")

Memuat data metadata dan konfigurasi...


NameError: name 'kgs' is not defined

# Single Planet Diagnostic Visualization

In [ ]:
# ==========================================
# TAHAP 3: SINGLE PLANET DIAGNOSTIC VISUALIZATION
# ==========================================

print("Menjalankan visualisasi diagnostik untuk satu planet sampel...")

# 1. Inisialisasi model prediksi khusus untuk visualisasi
# Kita menggunakan PredictionModel dari ariel_gp yang ada di repositori Juara 1
model_visualization = ariel_gp.PredictionModel()

# 2. Mengaktifkan fitur diagnostik
# Ini akan memerintahkan model untuk menampilkan plot hasil fitting
model_visualization.plot_final = True

# 3. Training/Inference 
# Meskipun model ini tidak memerlukan training yang kompleks untuk visualisasi, 
# kita tetap memanggil train() untuk memenuhi standar alur kerja repositori
model_visualization.train(train_data)

# 4. Melakukan inferensi (fitting) hanya pada planet pertama
# Kita menggunakan slice [0:1] untuk mengambil tepat satu planet dari train_data
print("Memproses visualisasi untuk planet pertama...")
model_visualization.infer(train_data[0:1])

print("Visualisasi selesai. Grafik diagnostik akan muncul di atas.")

# Full Baseline Model Definition & Training

In [ ]:
# ==========================================
# TAHAP 4: FULL BASELINE MODEL DEFINITION & TRAINING
# ==========================================

print("Memulai definis dan pelatihan model baseline...")

# 1. Mendefinisikan model baseline
# Fungsi ini membangun struktur model Bayesian yang digabungkan dengan 
# komponen PCA dan fudge factors sesuai metodologi Juara 1
model = ariel_model.baseline_model()

# 2. Melakukan Pelatihan (Training)
# Proses ini akan melatih komponen PCA untuk transit depth dan 
# menyesuaikan 'fudge factors' pada dataset pelatihan
# Catatan: Pada dataset penuh, proses ini memakan waktu sekitar 4 jam.
print("Proses pelatihan sedang berjalan (memerlukan waktu beberapa jam untuk dataset penuh)...")
model.train(train_data)

print("Pelatihan selesai. Model kini siap digunakan untuk tahap inferensi.")

# Inference and Scoring 

In [ ]:
# ==========================================
# TAHAP 5: INFERENCE, SCORING, & SUBMISSION
# ==========================================

print("Memulai tahap inferensi pada test data...")

# 1. Inferensi
# Metode .infer() akan menjalankan model yang sudah terlatih untuk memprediksi
# kedalaman transit pada test_data. Proses ini membaca sensor data secara paralel
# dan bisa memakan waktu beberapa jam.
inferred_data = model.infer(test_data)

# 2. Scoring (Hanya jika bukan saat submission resmi)
# Jika kodingan dijalankan secara lokal/di Kaggle bukan untuk submit resmi,
# kita bisa melihat metrik performa lokal untuk evaluasi.
if not kgs.is_submission:
    score = kgs.score_metric(inferred_data, test_data)
    print(f"Skor performa lokal: {score}")

# 3. Submission Generation
# Mengonversi data hasil inferensi ke dalam format dataframe submisi
df_submission = kgs.make_submission_dataframe(inferred_data)

# 4. Menyimpan ke CSV
# File 'submission.csv' akan tersimpan di direktori /kaggle/working/
kgs.write_submission_csv(df_submission)

print("Tahap inferensi selesai. File 'submission.csv' telah berhasil dibuat.")